In [ ]:
!pip install langchain-google-genai langgraph langchain-core

In [1]:
from langchain_google_genai import ChatGoogleGenerativeAI
import os

# Loading API key securely from Colab secrets
try:
    from google.colab import userdata
    os.environ["GOOGLE_API_KEY"] = userdata.get("GOOGLE_API_KEY")
    print("API key loaded from Colab secrets.")
except Exception:
    from dotenv import load_dotenv
    load_dotenv()
    print("Attempted to load API key from .env file.")

if not os.environ.get("GOOGLE_API_KEY"):
    raise RuntimeError("Missing GOOGLE_API_KEY — add it to Colab secrets or your .env file.")

model = ChatGoogleGenerativeAI(
    model="gemini-2.5-flash",
    temperature=0.3
)

print("Gemini model initialized successfully.")

API key loaded from Colab secrets.
Gemini model initialized successfully.


In [2]:
from langchain_core.messages import SystemMessage, HumanMessage, AIMessage, AnyMessage
from langgraph.graph import StateGraph, END
from langgraph.checkpoint.memory import MemorySaver
from typing import TypedDict, Annotated
import operator

ORDER_DATA = {
    "ORD-1001": {"product": "UltraPhone X",   "qty": 1, "status": "Shipped",    "delivery": "2025-08-10"},
    "ORD-1002": {"product": "SmartWatch Pro", "qty": 2, "status": "Processing", "delivery": "2025-08-14"},
    "ORD-1003": {"product": "TabletMax 12",   "qty": 1, "status": "Delivered",  "delivery": "2025-08-01"},
}

ORDER_STATUS_PROMPT = """
You are Alex, an Order Status specialist at Shopify customer support.
You help customers check their order status and delivery dates.
Available orders: {orders}
If asked about an order, look it up and provide product name, quantity, status, and delivery date.
If the order is not found, politely say so.
Always be concise, friendly, and reassuring.
""".format(orders=str(ORDER_DATA))

REFUND_POLICY_PROMPT = """
You are Jordan, a Refund Policy specialist at Shopify customer support.
You explain refund and return policies clearly. Key policy:
- Returns accepted within 30 days of purchase
- Items must be unused and in original packaging
- Refunds processed in 5-7 business days
- Digital products are non-refundable
- Customer needs their order number to start a return
Be empathetic and helpful.
"""

PRODUCT_SUGGESTION_PROMPT = """
You are Sam, a Product Recommendation specialist at Shopify.
You help customers find the right product. Available products:
- UltraPhone X: flagship smartphone, $999, best camera
- SmartWatch Pro: fitness & notifications, $299, 7-day battery
- TabletMax 12: 12-inch tablet, $649, great for students
- BudEars Wireless: noise-cancelling earbuds, $199
- PowerBank 20K: 20000mAh portable charger, $49
Ask about their budget and needs, then suggest 1-2 products with brief reasons.
"""

class AgentState(TypedDict):
    messages: Annotated[list[AnyMessage], operator.add]

def invoke_agent(state: AgentState, system_prompt: str) -> dict:
    messages = [SystemMessage(content=system_prompt)] + state["messages"]
    result = model.invoke(messages)
    return {"messages": [AIMessage(content=result.content)]}

def order_status_agent(state: AgentState) -> dict:
    """Agent 1: Handles order tracking inquiries."""
    clean_messages = state["messages"][:-1]
    messages = [SystemMessage(content=ORDER_STATUS_PROMPT)] + clean_messages
    result = model.invoke(messages)
    return {"messages": [AIMessage(content=result.content)]}

def refund_policy_agent(state: AgentState) -> dict:
    """Agent 2: Handles refund and return policy questions."""
    clean_messages = state["messages"][:-1]
    messages = [SystemMessage(content=REFUND_POLICY_PROMPT)] + clean_messages
    result = model.invoke(messages)
    return {"messages": [AIMessage(content=result.content)]}

def product_suggestion_agent(state: AgentState) -> dict:
    """Agent 3: Handles product recommendation requests."""
    clean_messages = state["messages"][:-1]
    messages = [SystemMessage(content=PRODUCT_SUGGESTION_PROMPT)] + clean_messages
    result = model.invoke(messages)
    return {"messages": [AIMessage(content=result.content)]}

print("Three customer service agents defined.")

Three customer service agents defined.


In [3]:
import uuid

class RouterAgent:

    def __init__(self, model, system_prompt, smalltalk_prompt, debug=False):
        self.system_prompt = system_prompt
        self.smalltalk_prompt = smalltalk_prompt
        self.model = model
        self.debug = debug

        router_graph = StateGraph(AgentState)

        router_graph.add_node("Router", self.call_llm)
        router_graph.add_node("Order_Agent", order_status_agent)
        router_graph.add_node("Refund_Agent", refund_policy_agent)
        router_graph.add_node("Product_Agent", product_suggestion_agent)
        router_graph.add_node("Small_Talk", self.respond_smalltalk)

        router_graph.add_conditional_edges(
            "Router",
            self.find_route,
            {
                "ORDER":     "Order_Agent",
                "REFUND":    "Refund_Agent",
                "PRODUCT":   "Product_Agent",
                "SMALLTALK": "Small_Talk",
                "END": END
            }
        )

        router_graph.add_edge("Order_Agent", END)
        router_graph.add_edge("Refund_Agent", END)
        router_graph.add_edge("Product_Agent", END)
        router_graph.add_edge("Small_Talk", END)

        router_graph.set_entry_point("Router")

        memory = MemorySaver()
        self.router_graph = router_graph.compile(checkpointer=memory)

    def call_llm(self, state: AgentState) -> dict:
        """Router node: classifies the user intent."""
        messages = state["messages"]
        if self.debug:
            print(f"Router received: {messages[-1].content}")
        messages = [SystemMessage(content=self.system_prompt)] + messages
        result = self.model.invoke(messages)
        if self.debug:
            print(f"Router decision: {result.content}")
        return {"messages": [result]}

    def respond_smalltalk(self, state: AgentState) -> dict:
        """Small talk node: handles greetings and general chat."""
        messages = [SystemMessage(content=self.smalltalk_prompt)] + state["messages"]
        result = self.model.invoke(messages)
        return {"messages": [AIMessage(content=result.content)]}

    def find_route(self, state: AgentState) -> str:
      """Reads the router output and returns the destination node."""
      last_message = state["messages"][-1]
      content = last_message.content
      if isinstance(content, list):
          for item in content:
              if isinstance(item, dict) and "text" in item:
                  content = item["text"]
                  break
              elif isinstance(item, str):
                  content = item
                  break
      content = str(content).strip().upper()
      first_word = content.split()[0] if content.split() else "END"

      valid = {"ORDER", "REFUND", "PRODUCT", "SMALLTALK", "END"}
      destination = first_word if first_word in valid else "END"

      if self.debug:
          print(f"Routing to: {destination}")
      return destination

print("RouterAgent class defined.")

RouterAgent class defined.


In [4]:
system_prompt = """
You are a Router that analyzes the customer's message and picks exactly ONE of these options:

ORDER     - If the query is about an order: order status, delivery date, order details.
REFUND    - If the query is about returns, refunds, or cancellations.
PRODUCT   - If the query is about product features, recommendations, or pricing.
SMALLTALK - If the message is small talk: greetings, goodbyes, general chat.
END       - Default for anything else.

Your output must be ONLY one word: ORDER, REFUND, PRODUCT, SMALLTALK, or END.
Do not add any explanation.
"""

smalltalk_prompt = """
You are a friendly customer service assistant.
Respond professionally to greetings, goodbyes, and general small talk.
Mention that you can help with: order status, refunds/returns, and product recommendations.
"""

router_agent = RouterAgent(model, system_prompt, smalltalk_prompt, debug=False)
print("Customer Service Chatbot initialized!")

Customer Service Chatbot initialized!


In [5]:
import uuid
import time

user_inputs = [
    "Hi there!",
    "What is the status of order ORD-1001?",
    "When will it be delivered?",
    "How do I return something if I don't like it?",
    "What's the return window?",
    "I'm looking for a good pair of earbuds under $250",
    "Does it have noise cancellation?",
    "Thanks, goodbye!",
]

config = {"configurable": {"thread_id": str(uuid.uuid4())}}

print("=" * 60)
print("Customer Service Chatbot — Demo")
print("=" * 60)

for input in user_inputs:
    print(f"----------------------------------------\nUSER : {input}")
    #Format the user message
    user_message = {"messages":[HumanMessage(input)]}
    #Get response from the agent
    ai_response = router_agent.router_graph.invoke(user_message,config=config)
    #Print the response
    print(f"\nAGENT : {ai_response['messages'][-1].content}")
    time.sleep(13)

Customer Service Chatbot — Demo
----------------------------------------
USER : Hi there!

AGENT : Hi there! Thanks for reaching out.

I'm here to help with a few things, such as checking your **order status**, assisting with **refunds/returns**, or providing **product recommendations**.

How can I help you today?
----------------------------------------
USER : What is the status of order ORD-1001?

AGENT : Thanks for reaching out!

Your order **ORD-1001** for the **UltraPhone X (qty: 1)** has been **Shipped** and is expected to be delivered by **2025-08-10**.
----------------------------------------
USER : When will it be delivered?

AGENT : Your UltraPhone X from order **ORD-1001** is expected to be delivered by **2025-08-10**.
----------------------------------------
USER : How do I return something if I don't like it?

AGENT : I understand you're thinking ahead about returns, and I'm happy to explain our policy clearly for you. We want you to be completely satisfied with your purch